# CP3 · Notebook 03 — Descripción básica

Aplicar Moondream2 a las 18 imágenes con un **prompt simple**: "describe esta escena de conducción". Comparar con expected_description. ~5 min.

In [ ]:
import time, json
from pathlib import Path
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA = Path('../datasets/dashcam-curated')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

MODEL_ID = 'vikhyatk/moondream2'
REVISION = '2024-08-26'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True,
                                               torch_dtype=torch.float32).eval()

## 1. Inferir sobre las 18 imágenes

In [ ]:
PROMPT = 'Describe this driving scene in 2-3 sentences. Mention any pedestrians, vehicles, traffic signs, weather conditions and notable hazards.'

categories = ['trivial', 'urbano_standard', 'edge_visual', 'edge_semantic', 'trampa', 'ambigua']
results = []

from tqdm import tqdm
all_imgs = []
for cat in categories:
    for p in sorted((DATA / cat).glob('*.jpg')) + sorted((DATA / cat).glob('*.png')):
        all_imgs.append((cat, p))

for cat, img_path in tqdm(all_imgs, desc='Moondream2'):
    img = Image.open(img_path).convert('RGB')
    t0 = time.perf_counter()
    enc = model.encode_image(img)
    description = model.answer_question(enc, PROMPT, tokenizer)
    elapsed = time.perf_counter() - t0
    results.append({
        'category': cat,
        'name': img_path.name,
        'path': str(img_path.relative_to(DATA)),
        'description': description,
        'time_s': elapsed,
    })

print(f'\nLatencia media: {sum(r["time_s"] for r in results)/len(results):.1f}s/imagen')
print(f'Latencia total: {sum(r["time_s"] for r in results):.0f}s')

## 2. Mostrar 6 ejemplos (1 por categoría)

In [ ]:
for cat in categories:
    cat_results = [r for r in results if r['category'] == cat]
    if not cat_results: continue
    r = cat_results[0]
    img = Image.open(DATA / r['path']).convert('RGB')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.imshow(img); ax1.set_title(f'{cat}: {r["name"]}'); ax1.axis('off')
    ax2.text(0.05, 0.95, r['description'], transform=ax2.transAxes,
             verticalalignment='top', wrap=True, fontsize=10,
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#F5F2EE'))
    ax2.set_title(f'Moondream2 ({r["time_s"]:.1f}s)')
    ax2.axis('off')
    plt.tight_layout(); plt.show()

## 3. Guardar resumen para 04/05/06

In [ ]:
with open(OUT / '03_descriptions.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'✅ {len(results)} descripciones guardadas en {OUT / "03_descriptions.json"}')

## 4. Reflexión rápida

Mira los 6 ejemplos. Apunta:

1. ¿En cuáles describe bien la escena? (suelen ser `trivial` y `urbano_standard`)
2. ¿En cuáles mete información que **NO está en la imagen**? (hallucination — buscar en `edge_visual`, `trampa`)
3. ¿Hay diferencias notables en **estilo** de respuesta entre categorías?

Compara con los `*_expected.md` de la carpeta dataset cuando estén rellenos. Para `07_local_vs_frontier.md` necesitarás esta comparación.

Ve a `04_tareas_estructuradas.ipynb`.